In [2]:
!pip install iterative-stratification

In [ ]:
# =============================
# Multi-Label Classification on NIH Chest X-ray Dataset
# =============================

import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models
import torch.optim as optim
import torch.nn.functional as F

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, precision_recall_curve
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from torch.cuda.amp import GradScaler, autocast
from torch.utils.tensorboard import SummaryWriter
import logging

# =============================
# 1. CONFIGURATION
# =============================
SEED = 42
BATCH_SIZE = 16
EPOCHS = 50  # Increased to allow more training epochs

# Modify to match your dataset location:
ROOT_DIR = "/kaggle/input/data"  # The folder that contains images_001, ..., train_val_list.txt, etc.
CSV_FILE = os.path.join(ROOT_DIR, "Data_Entry_2017.csv")
TRAIN_LIST_FILE = os.path.join(ROOT_DIR, "train_val_list.txt")
TEST_LIST_FILE  = os.path.join(ROOT_DIR, "test_list.txt")

# Define the diseases
DISEASES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule",
    "Pneumonia", "Pneumothorax", "Consolidation", "Edema", "Emphysema",
    "Fibrosis", "Pleural_Thickening", "Hernia", "No Finding"
]
NUM_LABELS = len(DISEASES)

# Set random seeds for reproducibility
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Logging setup
logging.basicConfig(level=logging.INFO, filename='training.log', 
                    format='%(asctime)s %(levelname)s:%(message)s')
logger = logging.getLogger()

# =============================
# 2. LOAD CSV + TRAIN/TEST LISTS
# =============================
if not os.path.exists(CSV_FILE):
    raise FileNotFoundError(f"{CSV_FILE} not found.")

# Load the NIH label CSV (Data_Entry_2017.csv)
df = pd.read_csv(CSV_FILE)
df = df[["Image Index", "Finding Labels"]]

# Create a dictionary: filename -> label string
labels_map = {}
for _, row in df.iterrows():
    filename = row["Image Index"]  # e.g. "00000003_000.png"
    label_str = row["Finding Labels"]  # e.g. "Cardiomegaly|Effusion"
    labels_map[filename] = label_str

# Load the train/val and test split files
def load_split_list(list_path):
    """Returns a list of image filenames from train_val_list.txt or test_list.txt."""
    if not os.path.exists(list_path):
        raise FileNotFoundError(f"{list_path} does not exist.")
    with open(list_path, 'r') as f:
        return [line.strip() for line in f if line.strip()]

train_val_filenames = load_split_list(TRAIN_LIST_FILE)
test_filenames      = load_split_list(TEST_LIST_FILE)

logger.info(f"Train/Val images: {len(train_val_filenames)}, Test images: {len(test_filenames)}")
print(f"Loaded {len(train_val_filenames)} training/validation images and {len(test_filenames)} test images.")

# =============================
# 3. DATASET CLASS
# =============================
class NIHChestXRayDataset(Dataset):
    def __init__(self, filenames, root_dir, labels_map, transform=None):
        """
        filenames: list of image filenames, e.g. ["00000003_000.png", ...]
        root_dir: folder containing images_00X/images/ and CSV
        labels_map: dict mapping filename -> label string
        transform: torchvision (or Albumentations) transforms
        """
        self.filenames = filenames
        self.root_dir = root_dir
        self.labels_map = labels_map
        self.transform = transform
        
        # Build a dictionary that maps "00000003_000.png" -> "full/path/to/image"
        self.path_map = {}
        # Look for subfolders: images_001, images_002, ... that each contain an 'images' folder
        subfolders = [d for d in os.listdir(root_dir) if d.startswith("images_")]
        
        for sf in subfolders:
            images_subdir = os.path.join(root_dir, sf, "images")
            if not os.path.isdir(images_subdir):
                continue
            # For each file in images_00X/images
            for fn in os.listdir(images_subdir):
                full_path = os.path.join(images_subdir, fn)
                self.path_map[fn] = full_path
        
        # Filter out filenames that do not exist in path_map
        initial_length = len(self.filenames)
        self.filenames = [fn for fn in self.filenames if fn in self.path_map]
        filtered_count = initial_length - len(self.filenames)
        if filtered_count > 0:
            logger.info(f"Filtered out {filtered_count} missing files.")
            print(f"Filtered out {filtered_count} missing files.")
        else:
            print("No missing files found.")
    
    def __len__(self):
        return len(self.filenames)
    
    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = self.path_map[filename]
        
        # Load image
        try:
            image = Image.open(img_path).convert("RGB")  # Converts grayscale to RGB by duplicating channels
        except Exception as e:
            logger.error(f"Error loading image {img_path}: {e}")
            # Return a tensor of zeros if image fails to load
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        
        # Apply transforms if any
        if self.transform:
            image = self.transform(image)
        
        # Convert "Finding Labels" => multi-label tensor
        label_str = self.labels_map.get(filename, "")  # e.g. "Effusion|Cardiomegaly"
        label_split = label_str.split("|") if label_str else []
        
        label_tensor = torch.zeros(NUM_LABELS, dtype=torch.float)
        for disease in label_split:
            if disease in DISEASES:
                idx_ = DISEASES.index(disease)
                label_tensor[idx_] = 1.0
            else:
                logger.warning(f"Unexpected disease label '{disease}' in file {filename}.")
        
        return image, label_tensor

# =============================
# 4. DATA PREPROCESSING & DATALOADERS
# =============================

# Define normalization based on ImageNet statistics
normalize = T.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])

# Data augmentation for training
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.1),
    T.ToTensor(),  # Convert PIL Image to Tensor before RandomErasing
    normalize,
    T.RandomErasing(p=0.1),  # Now correctly applied on Tensor
])

# No augmentation for validation and testing
test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    normalize,
])

# Multi-Label Stratified Train-Validation Split
logger.info("Starting multi-label stratified train-validation split...")
print("Performing multi-label stratified train-validation split...")

mlb = MultiLabelBinarizer(classes=DISEASES)
train_val_labels = [labels_map[fn].split("|") for fn in train_val_filenames]
binary_labels = mlb.fit_transform(train_val_labels)

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)

for train_idx, val_idx in msss.split(train_val_filenames, binary_labels):
    train_filenames = [train_val_filenames[i] for i in train_idx]
    val_filenames = [train_val_filenames[i] for i in val_idx]

logger.info(f"Training samples: {len(train_filenames)}, Validation samples: {len(val_filenames)}")
print(f"Training samples: {len(train_filenames)}, Validation samples: {len(val_filenames)}")

# Initialize Datasets
train_dataset = NIHChestXRayDataset(train_filenames, ROOT_DIR, labels_map, transform=train_transform)
val_dataset   = NIHChestXRayDataset(val_filenames, ROOT_DIR, labels_map, transform=test_transform)
test_dataset  = NIHChestXRayDataset(test_filenames, ROOT_DIR, labels_map, transform=test_transform)

# Handling class imbalance with weighted loss
# Calculate class weights based on training set
train_labels = [labels_map[fn].split("|") for fn in train_filenames]
binary_train_labels = mlb.transform(train_labels)
class_counts = binary_train_labels.sum(axis=0)
class_weights = 1.0 / (class_counts + 1e-6)  # Adding epsilon to avoid division by zero
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
logger.info(f"Class weights: {class_weights}")
print(f"Class weights: {class_weights}")

# Define Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        """
        alpha: weights for each class, shape [num_classes]
        gamma: focusing parameter
        reduction: 'mean' or 'sum'
        """
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # BCE loss
        BCE_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-BCE_loss)  # pt is the probability of being classified to the true class
        F_loss = (1 - pt) ** self.gamma * BCE_loss

        if self.alpha is not None:
            # alpha should be of shape [num_classes]
            alpha = self.alpha.unsqueeze(0)  # Shape [1, num_classes]
            F_loss = alpha * F_loss

        if self.reduction == 'mean':
            return F_loss.mean()
        elif self.reduction == 'sum':
            return F_loss.sum()
        else:
            return F_loss

# Initialize Focal Loss with class weights as alpha
focal_criterion = FocalLoss(alpha=class_weights, gamma=2, reduction='mean')
print("Initialized FocalLoss with class weights.")

# Initialize DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

logger.info(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}, Test samples: {len(test_dataset)}")
print(f"DataLoaders initialized with Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)} samples.")

# =============================
# 5. MODEL, LOSS, & TRAINING LOOP
# =============================
# Initialize the model (ResNet50) with pre-trained weights
model = models.resnet50(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, NUM_LABELS)
model = model.to(device)
print("Initialized ResNet50 model with modified final layer.")

# Define optimizer (AdamW is often better for fine-tuning)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
print("Initialized AdamW optimizer.")

# Define learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3, verbose=True)
print("Initialized ReduceLROnPlateau scheduler.")

# Initialize GradScaler for mixed precision training
scaler = GradScaler()
print("Initialized GradScaler for mixed precision training.")

# Initialize TensorBoard writer
writer = SummaryWriter()
print("Initialized TensorBoard SummaryWriter.")

# Define training function
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    for batch_idx, (images, labels) in enumerate(loader, 1):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
        if batch_idx % 50 == 0 or batch_idx == len(loader):
            print(f"Batch {batch_idx}/{len(loader)}: Loss={loss.item():.4f}")
    
    average_loss = running_loss / len(loader)
    return average_loss

# Define evaluation function with metrics
@torch.no_grad()
def evaluate_metrics(model, loader, criterion, threshold=0.5, verbose=True):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_outputs = []
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        all_labels.append(labels.cpu().numpy())
        all_outputs.append(torch.sigmoid(outputs).cpu().numpy())
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    # Calculate AUC for each class
    aucs = []
    for i in range(NUM_LABELS):
        if np.sum(all_labels[:, i]) == 0:
            continue  # Skip classes with no samples
        try:
            auc = roc_auc_score(all_labels[:, i], all_outputs[:, i])
            aucs.append(auc)
        except ValueError:
            logger.warning(f"ROC AUC score not defined for class index {i}.")
            if verbose:
                print(f"ROC AUC score not defined for class '{DISEASES[i]}'. Skipping.")
    
    mean_auc = np.mean(aucs) if aucs else 0
    
    # Binarize predictions with a threshold (e.g., 0.5)
    preds = (all_outputs >= threshold).astype(int)
    
    # Calculate F1, Precision, Recall
    f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    precision = precision_score(all_labels, preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, preds, average='macro', zero_division=0)
    
    avg_loss = total_loss / len(loader)
    
    if verbose:
        print(f"Validation Metrics - Loss: {avg_loss:.4f}, AUC: {mean_auc:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
    
    return avg_loss, mean_auc, f1, precision, recall

# Function to find best thresholds based on validation set
def find_best_thresholds(model, loader, criterion):
    model.eval()
    all_labels = []
    all_outputs = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_labels.append(labels.numpy())
            all_outputs.append(probs)
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    best_thresholds = []
    for i in range(NUM_LABELS):
        if np.sum(all_labels[:, i]) == 0:
            best_thresholds.append(0.5)  # Default threshold
            continue
        precision_vals, recall_vals, thresholds = precision_recall_curve(all_labels[:, i], all_outputs[:, i])
        f1_scores = 2 * (precision_vals * recall_vals) / (precision_vals + recall_vals + 1e-8)
        best_idx = np.argmax(f1_scores)
        best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        best_thresholds.append(best_threshold)
        logger.info(f"Best threshold for class '{DISEASES[i]}': {best_threshold:.4f}")
        print(f"Best threshold for class '{DISEASES[i]}': {best_threshold:.4f}")
    
    return best_thresholds

# Function to apply best thresholds and calculate metrics
def evaluate_with_best_thresholds(model, loader, criterion, best_thresholds):
    model.eval()
    total_loss = 0.0
    all_labels = []
    all_outputs = []
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        all_labels.append(labels.cpu().numpy())
        all_outputs.append(torch.sigmoid(outputs).cpu().numpy())
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    preds = np.zeros_like(all_outputs)
    for i in range(NUM_LABELS):
        preds[:, i] = (all_outputs[:, i] >= best_thresholds[i]).astype(int)
    
    # Calculate metrics
    # Calculate AUC for each class
    aucs = []
    for i in range(NUM_LABELS):
        if np.sum(all_labels[:, i]) == 0:
            continue  # Skip classes with no samples
        try:
            auc = roc_auc_score(all_labels[:, i], all_outputs[:, i])
            aucs.append(auc)
        except ValueError:
            logger.warning(f"ROC AUC score not defined for class index {i}.")
    
    mean_auc = np.mean(aucs) if aucs else 0
    
    # Calculate F1, Precision, Recall
    f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    precision = precision_score(all_labels, preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, preds, average='macro', zero_division=0)
    
    # Calculate class-wise metrics
    class_f1 = f1_score(all_labels, preds, average=None, zero_division=0)
    class_precision = precision_score(all_labels, preds, average=None, zero_division=0)
    class_recall = recall_score(all_labels, preds, average=None, zero_division=0)
    
    avg_loss = total_loss / len(loader)
    
    print(f"Test Metrics with Optimized Thresholds - Loss: {avg_loss:.4f}, AUC: {mean_auc:.4f}, F1: {f1:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}")
    logger.info(f"Test Set with Optimized Thresholds: Loss={avg_loss:.4f}, AUC={mean_auc:.4f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}")
    
    # Log class-wise metrics
    print("\nClass-wise Metrics:")
    for i, disease in enumerate(DISEASES):
        if np.sum(all_labels[:, i]) == 0:
            continue
        print(f"{disease}: F1={class_f1[i]:.4f}, Precision={class_precision[i]:.4f}, Recall={class_recall[i]:.4f}")
        logger.info(f"{disease}: F1={class_f1[i]:.4f}, Precision={class_precision[i]:.4f}, Recall={class_recall[i]:.4f}")
    
    return avg_loss, mean_auc, f1, precision, recall

# Function to visualize prediction probabilities
def plot_prediction_distribution(all_outputs, all_labels, disease_index, disease_name):
    plt.figure(figsize=(8,6))
    plt.hist(all_outputs[:, disease_index][all_labels[:, disease_index]==1], bins=50, alpha=0.6, label='Positive')
    plt.hist(all_outputs[:, disease_index][all_labels[:, disease_index]==0], bins=50, alpha=0.6, label='Negative')
    plt.title(f'Prediction Probabilities for {disease_name}')
    plt.xlabel('Probability')
    plt.ylabel('Frequency')
    plt.legend()
    plt.show()

# Function to visualize some predictions
def visualize_predictions(dataset, model, device, num_samples=5, thresholds=None):
    """
    Visualize a few predictions alongside true labels.
    """
    model.eval()
    for i in range(num_samples):
        img, label = dataset[i]
        img_input = img.unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(img_input)
            prob = torch.sigmoid(output).cpu().numpy()[0]
            if thresholds:
                pred = (prob >= thresholds).astype(int)
            else:
                pred = (prob >= 0.5).astype(int)
            pred_labels = [DISEASES[j] for j, val in enumerate(pred) if val == 1]
            true_labels = [DISEASES[j] for j, val in enumerate(label) if val == 1]
        
        # Convert tensor to numpy array and unnormalize for visualization
        img_np = img.permute(1, 2, 0).numpy()
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)
        
        plt.figure(figsize=(6,6))
        plt.imshow(img_np)
        plt.axis('off')
        plt.title(f"True: {true_labels}\nPredicted: {pred_labels}")
        plt.show()

# =============================
# 6. TRAINING LOOP WITH FOCAL LOSS
# =============================
# Adjusted training loop to use Focal Loss
patience = 7  # Increased patience
best_loss = float('inf')
counter = 0
early_stop = False

logger.info("Starting training...")
print("Starting training...")

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    logger.info(f"Epoch {epoch+1}/{EPOCHS}")
    
    train_loss = train_one_epoch(model, train_loader, optimizer, focal_criterion, scaler)
    print(f"Training Loss: {train_loss:.4f}")
    logger.info(f"Training Loss: {train_loss:.4f}")
    
    val_loss, val_auc, val_f1, val_precision, val_recall = evaluate_metrics(model, val_loader, focal_criterion, threshold=0.5, verbose=True)
    
    scheduler.step(val_loss)
    
    # Log metrics to TensorBoard
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Val', val_loss, epoch)
    writer.add_scalar('AUC/Val', val_auc, epoch)
    writer.add_scalar('F1/Val', val_f1, epoch)
    writer.add_scalar('Precision/Val', val_precision, epoch)
    writer.add_scalar('Recall/Val', val_recall, epoch)
    
    # Check for improvement
    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        logger.info("Model improved. Saving best model.")
        print("Model improved. Saving best model.")
    else:
        counter += 1
        logger.info(f"No improvement. Early stopping counter: {counter}/{patience}")
        print(f"No improvement. Early stopping counter: {counter}/{patience}")
        if counter >= patience:
            logger.info("Early stopping triggered.")
            print("Early stopping triggered.")
            early_stop = True
            break

writer.close()

# Load the best model
model.load_state_dict(torch.load('best_model.pth'))
logger.info("Loaded the best model from 'best_model.pth'.")
print("Loaded the best model from 'best_model.pth'.")

# =============================
# 7. FIND BEST THRESHOLDS
# =============================
logger.info("Finding best thresholds based on validation set...")
print("\nFinding best thresholds based on validation set...")
best_thresholds = find_best_thresholds(model, val_loader, focal_criterion)

# =============================
# 8. TESTING THE MODEL WITH BEST THRESHOLDS
# =============================
print("\nEvaluating model on test set with optimized thresholds...")
test_loss, test_auc, test_f1, test_precision, test_recall = evaluate_with_best_thresholds(
    model, test_loader, focal_criterion, best_thresholds
)

# =============================
# 9. OPTIONAL: VISUALIZE PREDICTIONS
# =============================
print("\nVisualizing sample predictions from the test set...")
visualize_predictions(test_dataset, model, device, num_samples=5, thresholds=best_thresholds)

# =============================
# 10. ADDITIONAL DIAGNOSTICS
# =============================

# Plot prediction distributions for each class
def plot_prediction_distributions(model, loader, device):
    model.eval()
    all_labels = []
    all_outputs = []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs).cpu().numpy()
            all_labels.append(labels.numpy())
            all_outputs.append(probs)
    
    all_labels = np.vstack(all_labels)
    all_outputs = np.vstack(all_outputs)
    
    for i, disease in enumerate(DISEASES):
        plt.figure(figsize=(8,6))
        plt.hist(all_outputs[all_labels[:, i]==1, i], bins=50, alpha=0.6, label='Positive')
        plt.hist(all_outputs[all_labels[:, i]==0, i], bins=50, alpha=0.6, label='Negative')
        plt.title(f'Prediction Probabilities for {disease}')
        plt.xlabel('Probability')
        plt.ylabel('Frequency')
        plt.legend()
        plt.show()

# Uncomment the following line to plot prediction distributions
# plot_prediction_distributions(model, val_loader, device)

Using device: cuda
Loaded 86524 training/validation images and 25596 test images.
Performing multi-label stratified train-validation split...
Training samples: 69200, Validation samples: 17324
No missing files found.
No missing files found.
No missing files found.
Class weights: tensor([1.5097e-04, 7.3206e-04, 1.4436e-04, 9.0695e-05, 3.0989e-04, 2.6553e-04,
        1.4265e-03, 4.7393e-04, 4.3821e-04, 9.0744e-04, 8.7873e-04, 9.9900e-04,
        5.5741e-04, 8.8496e-03, 2.4752e-05], device='cuda:0')
Initialized FocalLoss with class weights.
DataLoaders initialized with Train: 69200, Val: 17324, Test: 25596 samples.


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 181MB/s] 


Initialized ResNet50 model with modified final layer.
Initialized AdamW optimizer.
Initialized ReduceLROnPlateau scheduler.
Initialized GradScaler for mixed precision training.
Initialized TensorBoard SummaryWriter.
Starting training...

Epoch 1/50


/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
<ipython-input-4-927ad4e4cfee>:290: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
<ipython-input-4-927ad4e4cfee>:305: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Batch 50/4325: Loss=0.0000
Batch 100/4325: Loss=0.0000
Batch 150/4325: Loss=0.0000
Batch 200/4325: Loss=0.0000
Batch 250/4325: Loss=0.0000
Batch 300/4325: Loss=0.0000
Batch 350/4325: Loss=0.0000
Batch 400/4325: Loss=0.0000
Batch 450/4325: Loss=0.0000
Batch 500/4325: Loss=0.0000
Batch 550/4325: Loss=0.0000
Batch 600/4325: Loss=0.0000
Batch 650/4325: Loss=0.0000
Batch 700/4325: Loss=0.0001
Batch 750/4325: Loss=0.0000
Batch 800/4325: Loss=0.0001
Batch 850/4325: Loss=0.0000
Batch 900/4325: Loss=0.0000
Batch 950/4325: Loss=0.0000
Batch 1000/4325: Loss=0.0000
Batch 1050/4325: Loss=0.0000
Batch 1100/4325: Loss=0.0000
Batch 1150/4325: Loss=0.0000
Batch 1200/4325: Loss=0.0000
Batch 1250/4325: Loss=0.0000
Batch 1300/4325: Loss=0.0000
Batch 1350/4325: Loss=0.0000
Batch 1400/4325: Loss=0.0000
Batch 1450/4325: Loss=0.0000
Batch 1500/4325: Loss=0.0000
Batch 1550/4325: Loss=0.0000
Batch 1600/4325: Loss=0.0000
Batch 1650/4325: Loss=0.0000
Batch 1700/4325: Loss=0.0000
Batch 1750/4325: Loss=0.0000
Batch

...

Batch 50/4325: Loss=0.0000
Batch 100/4325: Loss=0.0000
Batch 150/4325: Loss=0.0000
Batch 200/4325: Loss=0.0000
Batch 250/4325: Loss=0.0000
Batch 300/4325: Loss=0.0000
Batch 350/4325: Loss=0.0000
Batch 400/4325: Loss=0.0000
Batch 450/4325: Loss=0.0000
Batch 500/4325: Loss=0.0000
Batch 550/4325: Loss=0.0000
Batch 600/4325: Loss=0.0000
Batch 650/4325: Loss=0.0000
Batch 700/4325: Loss=0.0000
Batch 750/4325: Loss=0.0000
Batch 800/4325: Loss=0.0000
Batch 850/4325: Loss=0.0000
Batch 900/4325: Loss=0.0000
Batch 950/4325: Loss=0.0000
Batch 1000/4325: Loss=0.0000
Batch 1050/4325: Loss=0.0000
Batch 1100/4325: Loss=0.0000
Batch 1150/4325: Loss=0.0000
Batch 1200/4325: Loss=0.0000
Batch 1250/4325: Loss=0.0000
Batch 1300/4325: Loss=0.0001
Batch 1350/4325: Loss=0.0000
Batch 1400/4325: Loss=0.0000
Batch 1450/4325: Loss=0.0000
Batch 1500/4325: Loss=0.0000
Batch 1550/4325: Loss=0.0000
Batch 1600/4325: Loss=0.0000
Batch 1650/4325: Loss=0.0000
Batch 1700/4325: Loss=0.0000
Batch 1750/4325: Loss=0.0000
Batch

<ipython-input-4-927ad4e4cfee>:305: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Batch 50/4325: Loss=0.0001
Batch 100/4325: Loss=0.0000
Batch 150/4325: Loss=0.0000
Batch 200/4325: Loss=0.0000
Batch 250/4325: Loss=0.0000
Batch 300/4325: Loss=0.0000
Batch 350/4325: Loss=0.0000
Batch 400/4325: Loss=0.0000
Batch 450/4325: Loss=0.0000
Batch 500/4325: Loss=0.0000
Batch 550/4325: Loss=0.0000
Batch 600/4325: Loss=0.0000
Batch 650/4325: Loss=0.0000
Batch 700/4325: Loss=0.0000
Batch 750/4325: Loss=0.0000
Batch 800/4325: Loss=0.0000
Batch 850/4325: Loss=0.0000
Batch 900/4325: Loss=0.0000
Batch 950/4325: Loss=0.0000
Batch 1000/4325: Loss=0.0000
Batch 1050/4325: Loss=0.0000
Batch 1100/4325: Loss=0.0000
Batch 1150/4325: Loss=0.0000
Batch 1200/4325: Loss=0.0000
Batch 1250/4325: Loss=0.0000
Batch 1300/4325: Loss=0.0000
Batch 1350/4325: Loss=0.0000
Batch 1400/4325: Loss=0.0000
Batch 1450/4325: Loss=0.0000
Batch 1500/4325: Loss=0.0000
Batch 1550/4325: Loss=0.0000
Batch 1600/4325: Loss=0.0000
Batch 1650/4325: Loss=0.0000
Batch 1700/4325: Loss=0.0000
Batch 1750/4325: Loss=0.0000
Batch

<ipython-input-4-927ad4e4cfee>:305: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Batch 50/4325: Loss=0.0000
Batch 100/4325: Loss=0.0000
Batch 150/4325: Loss=0.0000
Batch 200/4325: Loss=0.0000
Batch 250/4325: Loss=0.0000
Batch 300/4325: Loss=0.0000
Batch 350/4325: Loss=0.0000
Batch 400/4325: Loss=0.0000
Batch 450/4325: Loss=0.0000
Batch 500/4325: Loss=0.0000
Batch 550/4325: Loss=0.0000
Batch 600/4325: Loss=0.0000
Batch 650/4325: Loss=0.0000
Batch 700/4325: Loss=0.0000
Batch 750/4325: Loss=0.0000
Batch 800/4325: Loss=0.0000
Batch 850/4325: Loss=0.0001
Batch 900/4325: Loss=0.0000
Batch 950/4325: Loss=0.0000
Batch 1000/4325: Loss=0.0000
Batch 1050/4325: Loss=0.0000
Batch 1100/4325: Loss=0.0000
Batch 1150/4325: Loss=0.0000
Batch 1200/4325: Loss=0.0000
Batch 1250/4325: Loss=0.0000
Batch 1300/4325: Loss=0.0000
Batch 1350/4325: Loss=0.0000
Batch 1400/4325: Loss=0.0000
Batch 1450/4325: Loss=0.0000
Batch 1500/4325: Loss=0.0000
Batch 1550/4325: Loss=0.0000
Batch 1600/4325: Loss=0.0000
Batch 1650/4325: Loss=0.0000
Batch 1700/4325: Loss=0.0000
Batch 1750/4325: Loss=0.0000
Batch